In [ ]:
# Load base model + Stage A LoRA checkpoint

In [2]:
import os

BASE_DIR = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational"
CKPT_DIR = os.path.join(BASE_DIR, "checkpoints")

for root, dirs, files in os.walk(CKPT_DIR):
    for f in files:
        full_path = os.path.join(root, f)
        size_mb = os.path.getsize(full_path) / (1024 * 1024)
        print(f"{full_path}  ({size_mb:.1f} MB)")

C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\checkpoints\stage_a\latest.pt  (453.0 MB)
C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\checkpoints\stage_a\best\best.pt  (453.0 MB)


In [5]:
class GPTConfig:
    def __init__(self, vocab_size=50257, max_seq_len=512, embed_dim=512,
                 num_heads=8, num_layers=12, d_ff=2730, dropout=0.1, **kwargs):
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.num_layers = num_layers
        self.d_ff = d_ff
        self.dropout = dropout
        for k, v in kwargs.items():
            setattr(self, k, v)

In [6]:
import torch

BASE_DIR = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational"
BASE_CKPT = BASE_DIR + r"\model\base_model.pt"
STAGE_A_CKPT = BASE_DIR + r"\checkpoints\stage_a\best\best.pt"

def inspect(path, label):
    print(f"\n===== {label} =====")
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    print("Type:", type(ckpt))
    if isinstance(ckpt, dict):
        print("Top-level keys:", list(ckpt.keys()))
        for k, v in ckpt.items():
            if hasattr(v, "keys"):
                sub_keys = list(v.keys())
                print(f"  '{k}' -> dict with {len(sub_keys)} keys, sample:", sub_keys[:10])
            elif hasattr(v, "shape"):
                print(f"  '{k}' -> tensor, shape {v.shape}")
            else:
                print(f"  '{k}' -> {type(v)}: {v if not hasattr(v,'__len__') or len(str(v))<100 else '...'}")
    return ckpt

base_ckpt = inspect(BASE_CKPT, "BASE MODEL")
stage_a_ckpt = inspect(STAGE_A_CKPT, "STAGE A CHECKPOINT")


===== BASE MODEL =====
Type: <class 'dict'>
Top-level keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'scaler_state_dict', 'train_loss', 'val_loss', 'train_losses', 'val_losses', 'config']
  'epoch' -> <class 'int'>: 16
  'model_state_dict' -> dict with 148 keys, sample: ['token_embed.weight', 'blocks.0.ln1.weight', 'blocks.0.ln1.bias', 'blocks.0.ln2.weight', 'blocks.0.ln2.bias', 'blocks.0.attn.qkv_proj.weight', 'blocks.0.attn.out_proj.weight', 'blocks.0.attn.rope.inv_freq', 'blocks.0.attn.rope.cos_cached', 'blocks.0.attn.rope.sin_cached']
  'optimizer_state_dict' -> dict with 2 keys, sample: ['state', 'param_groups']
  'scheduler_state_dict' -> dict with 7 keys, sample: ['base_lrs', 'last_epoch', '_step_count', '_is_initial', '_get_lr_called_within_step', '_last_lr', 'lr_lambdas']
  'scaler_state_dict' -> dict with 5 keys, sample: ['scale', 'growth_factor', 'backoff_factor', 'growth_interval', '_growth_tracker']
  'train_loss' -> <class 'float'>: 3

In [8]:
import torch
import re

BASE_DIR = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational"
MERGED_OUT = BASE_DIR + r"\checkpoints\stage_a_merged\merged.pt"

LORA_RANK = 32
LORA_ALPHA = 64
scaling = LORA_ALPHA / LORA_RANK

stage_a_sd = stage_a_ckpt["model_state_dict"]
merged_sd = {}

lora_modules = set()
for k in stage_a_sd.keys():
    m = re.match(r"(.*)\.lora_A$", k)
    if m:
        lora_modules.add(m.group(1))

print(f"Found {len(lora_modules)} LoRA-wrapped modules")

for prefix in lora_modules:
    base_w = stage_a_sd[f"{prefix}.base.weight"]
    lora_A = stage_a_sd[f"{prefix}.lora_A"]
    lora_B = stage_a_sd[f"{prefix}.lora_B"]

    delta = (lora_B @ lora_A) * scaling
    merged_w = base_w + delta
    merged_sd[f"{prefix}.weight"] = merged_w

# Copy over everything else unchanged (non-LoRA layers), skipping raw lora/base sub-keys
handled_prefixes = lora_modules
for k, v in stage_a_sd.items():
    skip = any(k.startswith(p + ".") for p in handled_prefixes)
    if skip:
        continue
    merged_sd[k] = v

print(f"Merged state dict has {len(merged_sd)} keys")
print("Sample merged keys:", list(merged_sd.keys())[:10])

# Save in the same format as base_model.pt so it loads the same way
torch.save({
    "epoch": stage_a_ckpt["epoch"],
    "model_state_dict": merged_sd,
    "config": base_ckpt["config"],
}, MERGED_OUT)

print(f"Saved merged checkpoint to {MERGED_OUT}")

Found 24 LoRA-wrapped modules
Merged state dict has 148 keys
Sample merged keys: ['blocks.5.attn.out_proj.weight', 'blocks.0.attn.out_proj.weight', 'blocks.2.attn.out_proj.weight', 'blocks.3.attn.qkv_proj.weight', 'blocks.6.attn.qkv_proj.weight', 'blocks.10.attn.out_proj.weight', 'blocks.8.attn.out_proj.weight', 'blocks.4.attn.qkv_proj.weight', 'blocks.2.attn.qkv_proj.weight', 'blocks.11.attn.out_proj.weight']
Saved merged checkpoint to C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\checkpoints\stage_a_merged\merged.pt


In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [14]:
# Cell 7 equivalent — model architecture (paste as-is from your training notebook)
class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :])

    def forward(self, x, seq_len):
        return self.cos_cached[:, :, :seq_len, :], self.sin_cached[:, :, :seq_len, :]

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(q, k, cos, sin):
    q = (q * cos) + (rotate_half(q) * sin)
    k = (k * cos) + (rotate_half(k) * sin)
    return q, k

class SwiGLU(nn.Module):
    def __init__(self, dim, d_ff, dropout):
        super().__init__()
        self.w1 = nn.Linear(dim, d_ff, bias=False)
        self.w2 = nn.Linear(dim, d_ff, bias=False)
        self.w3 = nn.Linear(d_ff, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w3(F.silu(self.w1(x)) * self.w2(x)))

class CausalSelfAttention(nn.Module):
    def __init__(self, dim, num_heads, max_seq_len, dropout):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv_proj = nn.Linear(dim, dim * 3, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.rope = RotaryEmbedding(self.head_dim, max_seq_len)
        self.dropout = dropout

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv_proj(x).reshape(B, T, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rope(x, T)
        q, k = apply_rope(q, k, cos, sin)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.out_proj(out)

class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = CausalSelfAttention(dim, num_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(dim)
        self.ffn = SwiGLU(dim, d_ff, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class ENGLlmModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.token_embed = nn.Embedding(cfg["vocab_size"], cfg["embed_dim"])
        self.blocks = nn.ModuleList([
            TransformerBlock(cfg["embed_dim"], cfg["num_heads"], cfg["d_ff"], cfg["max_seq_len"], cfg["dropout"])
            for _ in range(cfg["num_layers"])
        ])
        self.ln_f = nn.LayerNorm(cfg["embed_dim"])
        self.lm_head = nn.Linear(cfg["embed_dim"], cfg["vocab_size"], bias=False)

    def forward(self, input_ids):
        x = self.token_embed(input_ids)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.lm_head(x)

print("Model class defined.")

Model class defined.


In [16]:
CONFIG = {
    "vocab_size": 50257,
    "max_seq_len": 512,
    "embed_dim": 512,
    "num_heads": 8,
    "num_layers": 12,
    "d_ff": 2730,
    "dropout": 0.1,

    "lora_r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0.05,
    "target_modules": ["qkv_proj", "out_proj"],
}

In [17]:
model = ENGLlmModel(CONFIG)
merged = torch.load(MERGED_OUT, map_location="cpu", weights_only=False)
missing, unexpected = model.load_state_dict(merged["model_state_dict"], strict=False)
print("Missing:", missing)
print("Unexpected:", unexpected)

Missing: []
Unexpected: []


In [1]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

Mon Jul  6 12:19:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.86                 Driver Version: 591.86         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5050      WDDM  |   00000000:01:00.0  On |                  N/A |
| 41%   58C    P1             78W /  130W |    7754MiB /   8151MiB |    100%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
# --- GPU info + timing check ---
import time, torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")
if torch.cuda.is_available():
    print("Total VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

# Time N optimizer steps using your real train_loader, model, optimizer
N_STEPS_TO_TIME = 20   # number of optimizer steps to time (not batches)

use_amp = CONFIG["device"] == "cuda"
amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
_scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and amp_dtype == torch.float16))

model.train()
optimizer.zero_grad()

if torch.cuda.is_available():
    torch.cuda.synchronize()
t0 = time.time()

steps_done = 0
for i, (input_ids, labels) in enumerate(train_loader):
    input_ids = input_ids.to(CONFIG["device"], non_blocking=True)
    labels = labels.to(CONFIG["device"], non_blocking=True)

    with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp):
        logits = model(input_ids)
        loss = compute_loss(logits, labels) / CONFIG["grad_accum_steps"]

    _scaler.scale(loss).backward()

    if (i + 1) % CONFIG["grad_accum_steps"] == 0:
        _scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, CONFIG["grad_clip"])
        _scaler.step(optimizer)
        _scaler.update()
        optimizer.zero_grad()
        steps_done += 1

    if steps_done >= N_STEPS_TO_TIME:
        break

if torch.cuda.is_available():
    torch.cuda.synchronize()
t1 = time.time()

elapsed = t1 - t0
print(f"\nTimed {steps_done} optimizer steps in {elapsed:.2f}s")
print(f"Time per optimizer step: {elapsed/steps_done:.3f}s")
print(f"Estimated time for 200 steps: {(elapsed/steps_done)*200/60:.2f} min")
print(f"Estimated time for one full epoch ({total_steps//CONFIG['epochs']} steps): {(elapsed/steps_done)*(total_steps//CONFIG['epochs'])/60:.1f} min")

GPU: NVIDIA GeForce RTX 5050
Total VRAM (GB): 8.55


NameError: name 'CONFIG' is not defined